In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


def clean_dataframe(df):
    """
    Cleans specified columns in a DataFrame by removing commas, dots, and newlines.

    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    columns (list): List of column names to clean.

    Returns:
    pd.DataFrame: Cleaned DataFrame.
    """
    columns = [col for col in df.columns.to_list() if col != 'verse']
    df_cleaned = df.copy()
    df_cleaned[columns] = (
        df_cleaned[columns]
        .replace({",": "", "\.": "", "\n": " "}, regex=True)  # Remove unwanted characters
        .apply(lambda x: x.str.lower())  # Convert to lowercase
    )
    return df_cleaned


def compute_embeddings(model, df, columns):
    """
    Computes multiple types of embeddings for specified columns in a DataFrame.

    - "average": Averages embeddings across multiple translations for each row.
    - "separate": Stores embeddings for each translation separately in a list.
    - "concatenated": Concatenates all embeddings into a single vector.
    - "max": Computes element-wise maximum pooling across all embeddings.

    Parameters:
    model (SentenceTransformer): Pretrained model for computing embeddings.
    df (pd.DataFrame): Input DataFrame with cleaned text.
    columns (list): List of column names containing text translations.

    Returns:
    pd.DataFrame: DataFrame with multiple types of embeddings stored.
    """
    avg_embeddings = []
    separate_embeddings = []
    concat_embeddings = []
    max_embeddings = []

    for _, row in df.iterrows():
        texts = [row[col] for col in columns if pd.notnull(row[col])]  # Collect non-null translations

        if texts:
            text_embeddings = [model.encode(text) for text in texts]  # Compute embeddings

            # Average embedding
            avg_embedding = np.mean(text_embeddings, axis=0)

            # Separate embeddings
            separate_embedding = text_embeddings

            # Concatenated embedding (flatten all vectors)
            concat_embedding = np.concatenate(text_embeddings, axis=0)

            # Max pooling embedding (element-wise max)
            max_embedding = np.max(text_embeddings, axis=0)

        else:
            # Default empty values
            embedding_dim = model.get_sentence_embedding_dimension()
            avg_embedding = np.zeros(embedding_dim)
            separate_embedding = []
            concat_embedding = np.zeros(len(columns) * embedding_dim)  # Concatenated size depends on num of translations
            max_embedding = np.zeros(embedding_dim)

        avg_embeddings.append(avg_embedding)
        separate_embeddings.append(separate_embedding)
        concat_embeddings.append(concat_embedding)
        max_embeddings.append(max_embedding)

    # Store different embedding types
    df["embedding_average"] = avg_embeddings
    df["embedding_separate"] = separate_embeddings
    df["embedding_concatenated"] = concat_embeddings
    df["embedding_max"] = max_embeddings

    return df


In [ ]:
import pandas as pd

file_path = "/content/drive/My Drive/Progetti/Commedia/Traduzioni/inferno/I/inferno_1.csv"
df = pd.read_csv(file_path)
df_cleaned = clean_dataframe(df)

In [ ]:
# Load the multilingual embedding model
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

output_csv = "df_with_embeddings.csv"  # Output file

# Define the columns to process (translations)
columns_to_embed = ["singleton", "musa", "kirkpatrick", "durling"]

df_embedded = compute_embeddings(model, df, columns_to_embed)


print(f"Saving embeddings to {output_csv}...")
df_embedded.to_csv(output_csv, index=False)

print("Processing complete. Embeddings saved successfully!")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saving embeddings to df_with_embeddings.csv...
Processing complete. Embeddings saved successfully!


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


import numpy as np
import pandas as pd

def load_embeddings(csv_file):
    """
    Loads a DataFrame containing text and stored embeddings.

    Parameters:
    csv_file (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame with embeddings converted back to numpy arrays or lists of numpy arrays.
    """
    df = pd.read_csv(csv_file)

    # Convert "average" embedding from string to numpy array
    df["embedding_average"] = df["embedding_average"].apply(
        lambda x: np.fromstring(x[1:-1], sep=" ") if isinstance(x, str) else np.zeros(384)
    )

    # Convert "separate" embeddings from stringified lists back to lists of numpy arrays
    df["embedding_separate"] = df["embedding_separate"].apply(
        lambda x: [np.fromstring(vec[1:-1], sep=" ") for vec in x.strip("[]").split(";") if vec]
        if isinstance(x, str) and x.strip() else []
    )

    # Convert "concatenated" embedding from string to numpy array
    df["embedding_concatenated"] = df["embedding_concatenated"].apply(
        lambda x: np.fromstring(x[1:-1], sep=" ") if isinstance(x, str) else np.zeros(384 * len(df.columns) - 1)
    )

    # Convert "max" embedding from string to numpy array
    df["embedding_max"] = df["embedding_max"].apply(
        lambda x: np.fromstring(x[1:-1], sep=" ") if isinstance(x, str) else np.zeros(384)
    )

    return df


def find_most_similar(input_text, df, model, top_n=5, mode="average"):
    """
    Computes semantic similarity between the input text and stored embeddings.

    Parameters:
    input_text (str): The user-provided text to search for.
    df (pd.DataFrame): The DataFrame with stored embeddings.
    model (SentenceTransformer): Model used to encode the input text.
    top_n (int): Number of top results to return.
    mode (str): The embedding type to use. Options:
        - "average": Single averaged embedding per row.
        - "separate": Multiple embeddings per row (best match selected).
        - "concatenated": Embeddings concatenated into a single vector.
        - "max": Element-wise max pooling embeddings.

    Returns:
    pd.DataFrame: Top N most similar verses with similarity scores.
    """
    input_embedding = model.encode(input_text)

    if mode == "average":
        similarities = cosine_similarity([input_embedding], np.vstack(df["embedding_average"]))[0]

    elif mode == "separate":
        similarities = []
        for emb_list in df["embedding_separate"]:
            if emb_list:  # Ensure the list is not empty
                max_similarity = max(cosine_similarity([input_embedding], np.vstack(emb_list))[0])
            else:
                max_similarity = 0.0
            similarities.append(max_similarity)

    elif mode == "concatenated":
        # Ensure the input embedding matches the concatenated vector size
        num_translations = len(df.iloc[0]["embedding_separate"])  # Get number of translations per row
        input_embedding_extended = np.tile(input_embedding, num_translations)  # Repeat input embedding

        # Ensure the input size matches stored concatenated embeddings
        if input_embedding_extended.shape[0] != df["embedding_concatenated"].iloc[0].shape[0]:
            raise ValueError(f"Dimension mismatch: input ({input_embedding_extended.shape[0]}) vs stored ({df['embedding_concatenated'].iloc[0].shape[0]})")

        similarities = cosine_similarity([input_embedding_extended], np.vstack(df["embedding_concatenated"]))[0]

    elif mode == "max":
        similarities = cosine_similarity([input_embedding], np.vstack(df["embedding_max"]))[0]

    else:
        raise ValueError("Invalid mode. Use 'average', 'separate', 'concatenated', or 'max'.")

    df["similarity"] = similarities

    # Sort by similarity and return top matches
    top_matches = df.sort_values(by="similarity", ascending=False).head(top_n)

    return top_matches[["dante", "singleton", "musa", "kirkpatrick", "durling", "similarity"]]


In [ ]:

# Load the precomputed embeddings
csv_file = "df_with_embeddings.csv"  # Path to your file
print("Loading stored embeddings...")
df_embedded = load_embeddings(csv_file)


Loading stored embeddings...


<ipython-input-8-2a1a64e9b6c3>:26: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  lambda x: [np.fromstring(vec[1:-1], sep=" ") for vec in x.strip("[]").split(";") if vec]
<ipython-input-8-2a1a64e9b6c3>:32: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  lambda x: np.fromstring(x[1:-1], sep=" ") if isinstance(x, str) else np.zeros(384 * len(df.columns) - 1)


In [ ]:
# User input
input_text = input("Enter a verse or phrase: ")

# Find most similar verses using different embeddings
df_avg = find_most_similar(input_text, df, model, top_n=5, mode="average")
df_sep = find_most_similar(input_text, df, model, top_n=5, mode="separate")
df_concat = find_most_similar(input_text, df, model, top_n=5, mode="concatenated")
df_max = find_most_similar(input_text, df, model, top_n=5, mode="max")

Enter a verse or phrase: i have been a man, not anymore


In [ ]:
df_avg

,dante,singleton,musa,kirkpatrick,durling,similarity
22,"Rispuosemi: “Non omo, omo già fui, e li parent...","“No, not a living man, though once I was,”\nhe...","“No longer living man, though once I was,”\nhe...",His answer came to me: ‘No man; a man\nI was i...,"He replied: “Not a man, I was formerly a man,\...",0.598283
11,"e non mi si partia dinanzi al volto, anzi 'mpe...","And it did not\ndepart from before my eyes, bu...","And, everywhere I looked, the beast was there\...",This never ceased to dance before my face.\nNo...,and it did not depart from before my face but\...,0.397058
8,"così l'animo mio, ch'ancor fuggiva, si volse a...",so my mind which was\nstill fleeing turned bac...,"so I, although my mind was turned to flee,\ntu...",so in my mind – my thoughts all fleeing still ...,"so my spirit, still fleeing, turned back to ga...",0.393106
0,Nel mezzo del cammin di nostra vita mi ritrova...,Midway in the journey of our life I found\nmys...,Midway along the journey of our life\nI woke t...,"At one point midway on our path in life,\nI ca...","In the middle of the journey of our life, I ca...",0.375364
20,"Mentre ch'i' rovinava in basso loco, dinanzi a...",While I was ruining down to the depth\nthere a...,"While I was rushing down to that low place,\nm...","As I went, ruined, rushing to that low,\nthere...","While I was falling down into a low place, bef...",0.352022


In [ ]:
df_sep

,dante,singleton,musa,kirkpatrick,durling,similarity
22,"Rispuosemi: “Non omo, omo già fui, e li parent...","“No, not a living man, though once I was,”\nhe...","“No longer living man, though once I was,”\nhe...",His answer came to me: ‘No man; a man\nI was i...,"He replied: “Not a man, I was formerly a man,\...",0.570148
19,"tal mi fece la bestia sanza pace, che, venendo...","such did that peaceless beast make me, as,\nco...","so she made me do, that relentless beast;\ncom...","So, now, with me. That brute which knows no pe...","so she made me, that restless beast, who, comi...",0.410763
0,Nel mezzo del cammin di nostra vita mi ritrova...,Midway in the journey of our life I found\nmys...,Midway along the journey of our life\nI woke t...,"At one point midway on our path in life,\nI ca...","In the middle of the journey of our life, I ca...",0.409703
11,"e non mi si partia dinanzi al volto, anzi 'mpe...","And it did not\ndepart from before my eyes, bu...","And, everywhere I looked, the beast was there\...",This never ceased to dance before my face.\nNo...,and it did not depart from before my face but\...,0.408064
8,"così l'animo mio, ch'ancor fuggiva, si volse a...",so my mind which was\nstill fleeing turned bac...,"so I, although my mind was turned to flee,\ntu...",so in my mind – my thoughts all fleeing still ...,"so my spirit, still fleeing, turned back to ga...",0.384901


In [ ]:
df_concat

,dante,singleton,musa,kirkpatrick,durling,similarity
22,"Rispuosemi: “Non omo, omo già fui, e li parent...","“No, not a living man, though once I was,”\nhe...","“No longer living man, though once I was,”\nhe...",His answer came to me: ‘No man; a man\nI was i...,"He replied: “Not a man, I was formerly a man,\...",0.544630
8,"così l'animo mio, ch'ancor fuggiva, si volse a...",so my mind which was\nstill fleeing turned bac...,"so I, although my mind was turned to flee,\ntu...",so in my mind – my thoughts all fleeing still ...,"so my spirit, still fleeing, turned back to ga...",0.366409
0,Nel mezzo del cammin di nostra vita mi ritrova...,Midway in the journey of our life I found\nmys...,Midway along the journey of our life\nI woke t...,"At one point midway on our path in life,\nI ca...","In the middle of the journey of our life, I ca...",0.363166
11,"e non mi si partia dinanzi al volto, anzi 'mpe...","And it did not\ndepart from before my eyes, bu...","And, everywhere I looked, the beast was there\...",This never ceased to dance before my face.\nNo...,and it did not depart from before my face but\...,0.350150
20,"Mentre ch'i' rovinava in basso loco, dinanzi a...",While I was ruining down to the depth\nthere a...,"While I was rushing down to that low place,\nm...","As I went, ruined, rushing to that low,\nthere...","While I was falling down into a low place, bef...",0.327269


In [ ]:
df_max

,dante,singleton,musa,kirkpatrick,durling,similarity
22,"Rispuosemi: “Non omo, omo già fui, e li parent...","“No, not a living man, though once I was,”\nhe...","“No longer living man, though once I was,”\nhe...",His answer came to me: ‘No man; a man\nI was i...,"He replied: “Not a man, I was formerly a man,\...",0.503358
0,Nel mezzo del cammin di nostra vita mi ritrova...,Midway in the journey of our life I found\nmys...,Midway along the journey of our life\nI woke t...,"At one point midway on our path in life,\nI ca...","In the middle of the journey of our life, I ca...",0.356993
8,"così l'animo mio, ch'ancor fuggiva, si volse a...",so my mind which was\nstill fleeing turned bac...,"so I, although my mind was turned to flee,\ntu...",so in my mind – my thoughts all fleeing still ...,"so my spirit, still fleeing, turned back to ga...",0.343830
20,"Mentre ch'i' rovinava in basso loco, dinanzi a...",While I was ruining down to the depth\nthere a...,"While I was rushing down to that low place,\nm...","As I went, ruined, rushing to that low,\nthere...","While I was falling down into a low place, bef...",0.325403
11,"e non mi si partia dinanzi al volto, anzi 'mpe...","And it did not\ndepart from before my eyes, bu...","And, everywhere I looked, the beast was there\...",This never ceased to dance before my face.\nNo...,and it did not depart from before my face but\...,0.321097
